Fig. 3C single cell genotyping

Paired-end raw fastq files were merged using FLASH prior to this analysis.

In [1]:
suppressPackageStartupMessages({
    library("Biostrings")
    library("stringr")
    library("dplyr")
    })

In [2]:
#working directory
WD <- "data/"

In [3]:
#T80K positive 36 samples + WT positive 34 samples
samples <- c(
"T1","T2","T3","T4","T5",
"T7","T8","T9","T10","T11",
"T13","T16","T17","T18","T19",
"T20","T21","T22","T23","T25",
"T26","T30","T32","T33","T34",
"T35","T36","T37","T38","T39",
"T40","T41","T42","T43","T44",
"T45",
"W3","W6","W7","W8",
"W9","W11","W12","W14","W15",
"W16","W17","W18","W19","W21",
"W22","W24","W25","W26","W27",
"W28","W29","W30","W32","W33",
"W34","W35","W36","W38","W39",
"W43","W44","W45","W46","W47"
)

In [12]:
# Create empty data.frame
rows <- c(
  "number of total reads",
  "% of reads that have up-downstream seq",
  "rank 1 seq",
  "% of rank 1 seq",
  "rank 2 seq",
  "% of rank 2 seq",
  "rank 3 seq",
  "% of rank 3 seq",
  "rank 4 seq",
  "% of rank 4 seq",
  "rank 5 seq",
  "% of rank 5 seq"
)

summary <- data.frame(matrix(NA, nrow = length(rows), ncol = length(samples)))
rownames(summary) <- rows
colnames(summary) <- samples

In [13]:
#upstream or downstream sequences (12nt each) of the editing window (±20 from the SNP position)

upstream_seq <- "GCTGGGCTGGCT"

downstream_seq <- "TGAGCGAAAAAG"

In [14]:
#counts sequence frequencies per sample
for (sample in samples){
    
    input <- paste0(WD, sample, ".extendedFrags.fastq.gz")
        
    #read FLASH-merged fastq.gz file
    df <- data.frame(readDNAStringSet(input, format="fastq", use.names=FALSE))
    colnames(df) <- c('sequence')
        
    #number of total reads
    total_reads <- nrow(df)
        
    #find upstream 12bp
    up <- data.frame(str_split_fixed(df$sequence, upstream_seq, 2))
    colnames(up) <- c("split_1", "split_2")
    up[up == ""] <- NA 
    up <- na.omit(up) #if the upstream_seq was not detected, remove
    
    #find downstream 12bp
    down <- data.frame(str_split_fixed(up$split_2, downstream_seq, 2))
    colnames(down) <- c("split_1", "split_2")
    down[down == ""] <- 
    down <- na.omit(down) #if the downstream_seq was not detected, remove

    df2 <- data.frame("sequence" = down$split_1)
    
    #number of reads that contain upstream and downstream seq
    filtered_reads <- nrow(df2)
    
    if (nrow(df2) > 0) {
        #make a table
        result <- data.frame(table(unlist(df2$sequence)))
        colnames(result) <- c("sequence", "frequency")
        result <- result[order(result$frequency, decreasing=TRUE),]
        result$ratio <- round(result$frequency/nrow(df2)*100, 2)
    
        #store data in summary
        summary["number of total reads",sample] = total_reads
        summary["% of reads that have up-downstream seq",sample] = round(filtered_reads/total_reads*100, 2)
        summary["rank 1 seq",sample] = as.character(result[1, "sequence"])
        summary["% of rank 1 seq",sample] = result[1, "ratio"]
        summary["rank 2 seq",sample] = as.character(result[2, "sequence"])
        summary["% of rank 2 seq",sample] = result[2, "ratio"]
        summary["rank 3 seq",sample] = as.character(result[3, "sequence"])
        summary["% of rank 3 seq",sample] = result[3, "ratio"]
        summary["rank 4 seq",sample] = as.character(result[4, "sequence"])
        summary["% of rank 4 seq",sample] = result[4, "ratio"]
        summary["rank 5 seq",sample] = as.character(result[5, "sequence"])
        summary["% of rank 5 seq",sample] = result[5, "ratio"]
        
        }  else {
        #put NA in summary if nrow(df2) = 0
        summary["number of total reads",sample] = total_reads
        summary["% of reads that have up-downstream seq",sample] = 0
        summary["rank 1 seq",sample] = NA
        summary["% of rank 1 seq",sample] = NA
        summary["rank 2 seq",sample] = NA
        summary["% of rank 2 seq",sample] = NA
        summary["rank 3 seq",sample] = NA
        summary["% of rank 3 seq",sample] = NA
        summary["rank 4 seq",sample] = NA
        summary["% of rank 4 seq",sample] = NA
        summary["rank 5 seq",sample] = NA
        summary["% of rank 5 seq",sample] = NA
        }
}

In [15]:
summary

,T1,T2,T3,T4,T5,T7,T8,T9,T10,T11,⋯,W34,W35,W36,W38,W39,W43,W44,W45,W46,W47
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
number of total reads,775867,962259,754623,706494,874388,720997,875177,628599,1011006,646553,⋯,679318,633485,616860,663168,788453,664885,459363,764940,749821,743297
% of reads that have up-downstream seq,96.3,97.55,96.84,97.52,94.86,94.15,93.71,54.08,97.42,97.66,⋯,97.8,96.54,97.01,65.25,95.48,97.4,97.45,97.77,97.38,97.35
rank 1 seq,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,⋯,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC
% of rank 1 seq,99.22,99.22,95.77,98.51,99.43,99.37,99.39,99.31,99.47,99.43,⋯,99.39,99.33,98.92,99.36,99.42,98.62,99.49,99.46,99.38,98.98
rank 2 seq,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTCCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,AAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCCTCATCTAGTTGTAAC,⋯,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCAGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTTTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATCGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGTGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC
% of rank 2 seq,0.12,0.17,3.11,0.86,0.07,0.1,0.12,0.11,0.03,0.04,⋯,0.06,0.07,0.12,0.05,0.05,0.23,0.05,0.07,0.06,0.12
rank 3 seq,GAATTGGGGGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,AAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,AAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,AAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTTCTCGTCATCTAGTTGTAAC,⋯,GAATTGGGAGAAATTCATCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCCGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGAAC,GAATTGGGAGAAATTCATCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCATCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTGTAAC,GAATTGGGAGAAATTCATCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAATTGTAAC,AAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCAGTTGTAAC
% of rank 3 seq,0.09,0.09,0.1,0.13,0.03,0.03,0.04,0.11,0.03,0.03,⋯,0.03,0.06,0.12,0.03,0.03,0.16,0.03,0.03,0.03,0.09
rank 4 seq,GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTATAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTTATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,AAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,AAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTTCTCTTCATCTAGTTGTAAC,⋯,AAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTATCTCTTCATCTAGTTGTAAC,GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTGTAAC,A

In [16]:
#call genotype if the top sequence is >=95%, otherwise "Mixed"

WT_seq <- "GAATTGGGAGAAATTCACCTGTCTCTTCATCTAGTTGTAAC"

T80K_seq <- "GAATTGGGAGAAATTCACCTTTCTCTTCATCTAGTTGTAAC"

for (sample in samples){
    
    if (summary["% of rank 1 seq",sample] >= 95 & summary["rank 1 seq",sample] == T80K_seq) {
        summary["genotype",sample] = "T80K"
    } else if (summary["% of rank 1 seq",sample] >= 95 & summary["rank 1 seq",sample] == WT_seq) {
        summary["genotype",sample] = "WT"
    } else {
        summary["genotype",sample] = "Mixed"
    }
}

In [25]:
# extract genotype row
geno <- as.character(unlist(summary["genotype", ]))
names(geno) <- colnames(summary)

#result of T80K positive cells
genotype_T <- geno[grep("^T", names(geno))]
table(genotype_T)

#result of WT positive cells
genotype_W <- geno[grep("^W", names(geno))]
table(genotype_W)

genotype_T
Mixed  T80K    WT 
    2    31     3 

genotype_W
Mixed  T80K    WT 
    1     1    32 

In [30]:
write.csv(summary_df, file = "summary_20bp.csv", row.names = TRUE)

In [29]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /software/cellgen/team205/si9/envs/Seurat/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

time zone: Europe/London
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
[1] dplyr_1.1.4         stringr_1.5.2       Biostrings_2.70.3  
[4] GenomeInfoDb_1.38.8 XVector_0.42.0      IRanges_2.36.0     
[7] S4Vectors_0.40.2    BiocGenerics_0.48.1

loaded via a namespace (and not attached):
 [1] crayon_1.5.3            vctrs_0.6.5             cli_3.6.5              
 [4] 